# Online Retail Data Analysis

## 1. Dataset Overview

This section provides an initial overview of the dataset, including its dimensions, columns, data types, and sample records.

In [1]:
import pandas as pd

df = pd.read_csv("data/OnlineRetail.csv", encoding = "latin1")

In [2]:
df.shape

(541909, 8)

In [3]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [4]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


## 2. Dataset Dimensions

We examine the number of unique invoices, products, customers, and countries represented in the dataset.

In [6]:
df["InvoiceNo"].nunique()

25900

In [7]:
df["StockCode"].nunique()

4070

In [8]:
df["CustomerID"].nunique()

4372

In [9]:
df["Country"].nunique()

38

## 3. Transaction Structure

We investigate how products and customers are represented within individual invoices and examine the structure of transactions.

- How many Products can an Invoice have?

In [10]:
df.groupby("InvoiceNo")["StockCode"].nunique().max()

np.int64(1110)

In [11]:
df.groupby("InvoiceNo")["StockCode"].nunique().idxmax()

'573585'

In [12]:
df[df["InvoiceNo"] == "573585"]["StockCode"].duplicated().sum()

np.int64(4)

In [13]:
df.loc[~df["InvoiceNo"].str.isnumeric(), "InvoiceNo"].nunique()

3839

- Can an invoice be associated with multiple customers?

In [14]:
df[df["InvoiceNo"] == "536365"]["CustomerID"].nunique()

1

In [15]:
df.groupby("InvoiceNo")["CustomerID"].nunique().max()

np.int64(1)

## 4. Missing Values

In [16]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [17]:
df.isna().sum() / df.shape[0] * 100

InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
dtype: float64

24.92% of the dataset would be lost if we removed rows with missing CustomerID

In [18]:
df[df["CustomerID"].isna()]["InvoiceNo"]

622       536414
1443      536544
1444      536544
1445      536544
1446      536544
           ...  
541536    581498
541537    581498
541538    581498
541539    581498
541540    581498
Name: InvoiceNo, Length: 135080, dtype: object

- Are missing CustomerID associated with specified products?

In [19]:
df[df["CustomerID"].isna()]["InvoiceNo"].str.startswith("C").sum()

np.int64(383)

Only a small number of records with missing CustomerID have an InvoiceNo starting with "C", suggesting that missing CustomerIDs are not primarily associated with these invoices.

- Are missing descriptions associated with specific products?

In [26]:
df[df["Description"].isna()]["StockCode"].nunique()

960

There are 1,454 rows with missing descriptions, corresponding to 960 unique products.

- Do the same StockCodes have descriptions in other transactions?

In [36]:
description_counts = df.groupby("StockCode")["Description"].count()

missing_codes = df[df["Description"].isna()]["StockCode"]

description_counts[
    missing_codes.isin(description_counts.index)
    &
    (description_counts > 0)
]

description_counts.nunique()

617

617 out of 960 products with missing descriptions have a non-missing description in other transactions.

- Does each StockCode have a consistent Description?

In [45]:
df.groupby("StockCode")["Description"].nunique()

StockCode
10002           1
10080           2
10120           1
10123C          1
10123G          0
               ..
gift_0001_20    2
gift_0001_30    1
gift_0001_40    1
gift_0001_50    1
m               1
Name: Description, Length: 4070, dtype: int64

- Do products have multiple different descriptions?

In [46]:
df.groupby("StockCode")["Description"].nunique().max()

np.int64(8)

The same StockCode can have multiple different descriptions, so missing descriptions cannot be reliably filled using StockCode alone.

## 5. Data Cleaning

- Are there duplicate rows in the dataset?

In [ ]:
df.duplicated().sum()

np.int64(5268)

- What percentage of the dataset consists of duplicate rows?

In [54]:
df.duplicated().sum() / df.shape[0] * 100

np.float64(0.9721189350979592)

- What do duplicated rows look like?

In [57]:
df[df.duplicated()].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/2010 11:45,1.25,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,12/1/2010 11:45,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,12/1/2010 11:45,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/2010 11:45,4.95,17908.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,12/1/2010 11:49,2.95,17920.0,United Kingdom


The dataset contains 5,268 completely duplicated rows, representing about 0.97% of all rows.
Since these rows contain exactly the same information as previous records, they can be removed without losing unique information.

In [58]:
df.shape

(541909, 8)

In [59]:
df = df.drop_duplicates()

In [60]:
df.shape

(536641, 8)